In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print("Environment set. Now safe to import.")

Environment set. Now safe to import.


In [2]:
import sys
import numpy as np
from tqdm import tqdm
import faiss
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from typing import Optional

sys.path.insert(0, "../src/")
from embedder import encode, load_model
from schema import Document, Chunk
from cadec_loader import load_cadec
from chunker import chunk_document
load_model()

/Users/sakshi/Desktop/drug_safety_rag/drug-safety-rag/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Loading tokenizer...
Loading model...
Model loaded on cpu.


In [3]:
# reload chunks
docs = load_cadec("../data/cadec")
all_chunks=[]
for doc in docs:
    all_chunks.extend(chunk_document(doc, overlap=1, chunk_size=400))

embeddings = np.load("../index/embeddings.npy")
print(f"Chunk size: {len(all_chunks)}")
print(f"Embedding shape: {embeddings.shape}")
print(f"Match: {len(all_chunks)==embeddings.shape[0]}")

Loaded 1220 documents from ../data/cadec
Chunk size: 2549
Embedding shape: (2549, 768)
Match: True


In [4]:
# build faiss index

EMBEDDING_DIM =768

# create the index
index = faiss.IndexFlatL2(EMBEDDING_DIM)

# faiss accepts float vectors
vectors = embeddings.astype(np.float32)

# adding vectors to index
index.add(vectors)

print(f"Index type: {type(index)}")
print(f"Vector size: {index.ntotal}")
print(f"Dimension: {index.d}")


Index type: <class 'faiss.swigfaiss.IndexFlatL2'>
Vector size: 2549
Dimension: 768


In [5]:
# save the index so you don't have to every session
faiss.write_index(index, "../index/cadec.faiss")
print("Index saved -> '../index/cadec.faiss'")

Index saved -> '../index/cadec.faiss'


In [6]:
# load the saved index to verify
index_loaded = faiss.read_index("../index/cadec.faiss")
print(f"Loaded index: {index_loaded.ntotal} vectors")
print(f"Match: {index.ntotal==index_loaded.ntotal}")

Loaded index: 2549 vectors
Match: True


In [28]:
class ChunkStore:
    """Maps FAISS integer indices back to Chunk objects."""

    def __init__(self):
        self._chunks= []

    def add(self, chunk: Chunk)->int:
        idx = len(self._chunks)
        chunk.vector_index= idx
        self._chunks.append(chunk)
        return idx

    def get(self, vector_index: int)-> Chunk:
        return self._chunks[vector_index]

    def __len__(self):
        return len(self._chunks)

    def filter_by_drug(self, drug_name: str)->list:
        return [c for c in self._chunks if c.drug_name.lower()==drug_name.lower()]

        

In [29]:
store = ChunkStore()
for chunk in all_chunks:
    store.add(chunk)

print(f"ChunkStore: {len(store)} chunks")
print(f"Sample    : {store.get(1)}")

ChunkStore: 2549 chunks
Sample    : Chunk(id=7a12534a3746, doc=3deaf7e06301, src='cadec', drug='diclofenac', idx=1, vec=1, preview='Due to my arthritis getting progressively worse, t')


In [30]:
# def retrieve(query: str, k:int=5)->list:
#     """
#     Given a plain text query, return the k most relevant chunks.
#     """
#     # Step 1 — encode the query using the same pipeline as the chunks
#     query_vector = encode([query], batch_size=32)  # [1,768]
#     query_vector = query_vector.astype(np.float32) # [1,768]

#     # Step 2 - seach the index
#     distances, indices = index.search(query_vector, k)
#     # distances: [1, k] — lower = more similar (L2 distance)
#     # indices:   [1, k] — positions in the vector store

#     # Step 3 - map indices back to chunk objects
#     results = []
#     for dis, idx in zip(distances[0], indices[0]):
#         chunk= store.get(idx)
#         results.append({
#             "chunk": chunk,
#             "distance": float(dis),
#             "score": (1/(1+float(dis))) # convert distance → similarity score
#         })

#     return results


DRUG_KEYWORDS = {
    "atorvastatin": ["atorvastatin", "lipitor", "statin", "cholesterol", "ldl"],
    "diclofenac":   ["diclofenac", "voltaren", "arthrotec", "cataflam", 
                     "nsaid", "arthritis", "inflammation"],
}

def detect_drug(query: str)-> Optional[str]:
    query = query.lower()
    for drug, keywords in DRUG_KEYWORDS.items():
        if any(kw in query for kw in keywords):
            return drug
    return None # query everything if no drug found

def retrieve(query_vector: np.ndarray, query_text : str, k:int=5)->list:

    detected_drug = detect_drug(query_text)

    if detected_drug:
        # Filter chunk store to only this drug's chunks
        candidate_chunks = store.filter_by_drug(detected_drug)
        candidate_indices = [c.vector_index for c in candidate_chunks]

        # Search only within those vectors
        candidate_vectors = embeddings[candidate_indices].astype(np.float32)

        # Brute force over the subset — fast enough for this size
        from numpy.linalg import norm
        scores = candidate_vectors@query_vector[0] # dot product - cosine similarity
        top_k_local = np.argsort(scores)[::-1][:k]

        results= []
        for local_idx in top_k_local:
            global_idx = candidate_indices[local_idx]
            chunk = store.get(global_idx)
            results.append({
                "chunk": chunk,
                "score": scores[local_idx],
                "drug": detected_drug
            })
        return results

    else:
        # Step 2 - seach the index
        distances, indices = index.search(query_vector, k)
        # distances: [1, k] — lower = more similar (L2 distance)
        # indices:   [1, k] — positions in the vector store

        # Step 3 - map indices back to chunk objects
        results = []
        for dis, idx in zip(distances[0], indices[0]):
            chunk= store.get(idx)
            results.append({
                "chunk": chunk,
                "score": (1/(1+float(dis))), # convert distance → similarity score
                "drug": None
            })

        return results

In [31]:
def show_results(query_text: str, k: int=5):
    query_vector = encode([query_text], batch_size=32)  # [1,768]
    query_vector = query_vector.astype(np.float32) # [1,768]
     
    print(f"Query: '{query_text}'")
    print("="*60)

    results = retrieve(query_vector,query_text, k)
    for i, r in enumerate(results):
        chunk = r["chunk"]
        filter_note = f" [filtered to {r['drug']}]" if r['drug'] else ""
        print(f"\nRank {i+1} | score={r['score']:.4f} | drug={chunk.drug_name}{filter_note}")
        print(f"\n{chunk.text[:150]}")


        

In [32]:
# Test 1 — specific ADR
show_results("What are the muscle side effects of atorvastatin?")

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 20.53it/s]

Query: 'What are the muscle side effects of atorvastatin?'

Rank 1 | score=0.9777 | drug=atorvastatin [filtered to atorvastatin]

It has done a good job of lowering my cholestoral but, are the side effects worth it?.

Rank 2 | score=0.9770 | drug=atorvastatin [filtered to atorvastatin]

After 4 uneventful years on Lipitor, I started having intense shoulder and upper arm pain.
If I don't stop the Lipitor, what will happen?
Is this life

Rank 3 | score=0.9763 | drug=atorvastatin [filtered to atorvastatin]

Do the benefits outweigh the severe side effects-Pfizer says it's rare but serious side effects-I know many people that have had the same problem with

Rank 4 | score=0.9759 | drug=atorvastatin [filtered to atorvastatin]

Works just fine.
If there are any side effects, they are definitely not noticeable.
What's with all these older people (70's)complaining about the lac

Rank 5 | score=0.9752 | drug=atorvastatin [filtered to atorvastatin]

Simply don't know what to do to bring LDL down


/var/folders/0l/p3pw_5l94xs5nb6tm8p9nkp40000gn/T/ipykernel_3462/2638367042.py:54: RuntimeWarning: divide by zero encountered in matmul
  scores = candidate_vectors@query_vector[0] # dot product - cosine similarity
/var/folders/0l/p3pw_5l94xs5nb6tm8p9nkp40000gn/T/ipykernel_3462/2638367042.py:54: RuntimeWarning: overflow encountered in matmul
  scores = candidate_vectors@query_vector[0] # dot product - cosine similarity
/var/folders/0l/p3pw_5l94xs5nb6tm8p9nkp40000gn/T/ipykernel_3462/2638367042.py:54: RuntimeWarning: invalid value encountered in matmul
  scores = candidate_vectors@query_vector[0] # dot product - cosine similarity


In [ ]:
# Run this in a fresh cell before show_results
import psutil
ram = psutil.virtual_memory()
print(f"Total RAM : {ram.total / 1e9:.1f} GB")
print(f"Available : {ram.available / 1e9:.1f} GB")
print(f"Used      : {ram.used / 1e9:.1f} GB")


In [ ]:
import torch
print(torch.backends.mps.is_available())  # True on M1/M2 Mac
print(torch.version.__version__)

In [33]:
show_results("Does diclofenac cause any heart problems?") 
             


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 20.51it/s]

Query: 'Does diclofenac cause any heart problems?'

Rank 1 | score=0.9776 | drug=diclofenac [filtered to diclofenac]

My orthopaedic surgeon prescribes the Cataflam. It has a slight effect on my blood pressure, but, since my pressure is usually low, that is not consid

Rank 2 | score=0.9775 | drug=diclofenac [filtered to diclofenac]

was given this to replace ibuprofen (possible sun sensitivity developed with ibu) but it doesn't relieve any pain for me or reduce inflammation.

Rank 3 | score=0.9761 | drug=diclofenac [filtered to diclofenac]

Else how much would you have to take, thus increasing your risk for possible adverse reactions.

Rank 4 | score=0.9756 | drug=diclofenac [filtered to diclofenac]

i notice there are a bunch of random ppl who gave the medicine a score of 5 without any explanation.
I wouldn't be surprised if that was the pharmolog

Rank 5 | score=0.9754 | drug=diclofenac [filtered to diclofenac]

I dont think anything will completely remove the pain, and maybe should


/var/folders/0l/p3pw_5l94xs5nb6tm8p9nkp40000gn/T/ipykernel_3462/2638367042.py:54: RuntimeWarning: divide by zero encountered in matmul
  scores = candidate_vectors@query_vector[0] # dot product - cosine similarity
/var/folders/0l/p3pw_5l94xs5nb6tm8p9nkp40000gn/T/ipykernel_3462/2638367042.py:54: RuntimeWarning: overflow encountered in matmul
  scores = candidate_vectors@query_vector[0] # dot product - cosine similarity
/var/folders/0l/p3pw_5l94xs5nb6tm8p9nkp40000gn/T/ipykernel_3462/2638367042.py:54: RuntimeWarning: invalid value encountered in matmul
  scores = candidate_vectors@query_vector[0] # dot product - cosine similarity


In [34]:
show_results("I am experiencing memory loss")


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 21.98it/s]

Query: 'I am experiencing memory loss'

Rank 1 | score=0.9505 | drug=atorvastatin

Makes me very tired.

Rank 2 | score=0.9491 | drug=atorvastatin

Extreme fatigue and general weak feeling overall like I have never experienced EVER in my entire life before.
In all honestly, I feel as though my sen

Rank 3 | score=0.9474 | drug=atorvastatin

I also am experiencing increased memory loss, and fatigue, but then I rarely sleep well (due to night sweats) since I went through menopause ten years

Rank 4 | score=0.9473 | drug=atorvastatin

Within 1 month time developed severe depression, inability to pull myself from bed, headaches everyday, and felt like I was coming down with flu.
I wi

Rank 5 | score=0.9471 | drug=diclofenac

Now I dont know how I am going to get back to work, it has left me feeling exausted, and depressed.


In [35]:
show_results("Is atorvastatin effective at lowering cholesterol?")

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 17.72it/s]

Query: 'Is atorvastatin effective at lowering cholesterol?'

Rank 1 | score=0.9741 | drug=atorvastatin [filtered to atorvastatin]

Higher cholesterol in those over 60 is now found to be PROTECTIVE particularly against stroke!.

Rank 2 | score=0.9735 | drug=atorvastatin [filtered to atorvastatin]

Taking 40mg daily for several years.
Cholesterol much lower,HDL higher,LDL below 100 and lower triglycerides.

Rank 3 | score=0.9727 | drug=atorvastatin [filtered to atorvastatin]

Simply don't know what to do to bring LDL down. At this stage it is not clear what is better of the two evils: Lipitor or High LDL? And how to lower L

Rank 4 | score=0.9724 | drug=atorvastatin [filtered to atorvastatin]

If you read the research carefully, there is reason to believe that higher levels of cholesterol are associated with lower overall mortality rather th

Rank 5 | score=0.9720 | drug=atorvastatin [filtered to atorvastatin]

Am definitely thinking of going off lipids since doing some research. People 


/var/folders/0l/p3pw_5l94xs5nb6tm8p9nkp40000gn/T/ipykernel_3462/2638367042.py:54: RuntimeWarning: divide by zero encountered in matmul
  scores = candidate_vectors@query_vector[0] # dot product - cosine similarity
/var/folders/0l/p3pw_5l94xs5nb6tm8p9nkp40000gn/T/ipykernel_3462/2638367042.py:54: RuntimeWarning: overflow encountered in matmul
  scores = candidate_vectors@query_vector[0] # dot product - cosine similarity
/var/folders/0l/p3pw_5l94xs5nb6tm8p9nkp40000gn/T/ipykernel_3462/2638367042.py:54: RuntimeWarning: invalid value encountered in matmul
  scores = candidate_vectors@query_vector[0] # dot product - cosine similarity
